# ILCOF-1 | Configurable 2D workflow

This notebook demonstrates the COF-Landscaper workflow for ILCOF-1. It covers framework construction, screening of 28 serrated layer arrangements, relaxation of the lowest-energy sampled structure and comparison of its simulated powder X-ray diffraction (PXRD) pattern with experiment.

Start Jupyter with `configurable/` as the notebook’s working directory. It contains its own molecular inputs and experimental scan. Calculations create the numbered stages under `ILCOF-1/`. To revisit a stage with its input files already on disk, run the import, system inputs and calculator setup first. Use a fresh working directory when comparing different grids or calculators so earlier results stay identifiable.

In [ ]:
import coflandscaper as cl
from mace.calculators import mace_mp

## System inputs

`COF_NAME` names the structures and their output tree. `TOPOLOGY` selects the connectivity of the layer. Here `sql` is the four-connected square-lattice net.

In [ ]:
COF_NAME = "ILCOF-1"
TOPOLOGY = "sql"

## Choose the energy model

A machine-learning interatomic potential estimates the energy of a structure. ASE uses the calculator’s forces and stress to relax atomic positions and the cell. In MACE-MH-1, the **head** selects the training reference for the energy surface. The **device** and **precision** control where and how the numerical calculation runs.

| Heads in the distributed checkpoint used here | Reference and COF-Landscaper dispersion treatment |
| --- | --- |
| `spice_wB97M` | Molecular ωB97M-D3(BJ) reference. Dispersion contributes to the training energies. No separate D3 term is added. |
| `omol` | Molecular ωB97M-V reference including nonlocal correlation. No separate D3 term is added. |
| `omat_pbe`, `matpes_r2scan` | Materials heads. The package adds D3 using PBE and r²SCAN parameters, respectively. |
| `mp_pbe_refit_add`, `oc20_usemppbe` | Materials Project refit and OC20 adsorption heads. The preset leaves an additional dispersion correction off. |

The default here is `spice_wB97M` on CPU in double precision. The first setup may download the model. With the setup below, `calculator.head` and `calculator.available_heads` give the selected head and the choices in the loaded checkpoint. A compatible NVIDIA GPU with CUDA-enabled PyTorch can use `device="cuda"`. Native AMD GPU execution has yet to be validated for this workflow. [MACE-MH-1 model card](https://huggingface.co/mace-foundations/mace-mh-1)

Create one ASE calculator with MACE’s `mace_mp` factory. Here `dispersion=False` matches the package preset for `spice_wB97M`. If you change the head in this direct call, set any separate D3 correction and its parameters explicitly. `SinglePoint` uses the shared calculator to evaluate fixed structures. `GeometryOpt` uses its forces and stress to relax them. To use another ASE calculator, assign an instance providing energy, forces and stress to `calculator` in this cell. [ASE calculators](https://docs.ase-lib.org/ase/calculators/calculators.html)

In [ ]:
calculator = mace_mp(
    model="mh-1",
    head="spice_wB97M",
    device="cpu",
    default_dtype="float64",
    dispersion=False,
)

**Optional ASE calculator: UMA.** With `fairchem-core` and access to the UMA checkpoint, the following can replace the MACE setup cell. The `omat` task illustrates a periodic bulk-material calculator. Single-point calculations require energy, atomic relaxation also requires forces, and cell relaxation requires stress. Suitability for ILCOF-1 requires separate scientific validation. [FAIR Chemistry example](https://facebookresearch.github.io/fairchem/formation-energy/)

```python
from fairchem.core.calculate import FAIRChemCalculator

calculator = FAIRChemCalculator.from_model_checkpoint(
    "uma-s-1p2p1", task_name="omat"
)
```

## Build the single layer

The **node**, derived from 1,3,6,8-tetrakis(4-formylphenyl)pyrene, joins four neighbours. Each p-phenylenediamine-derived **linker** bridges two nodes. The supplied XYZs represent prepared framework fragments. Each He marker designates a connection site and guides the fragment orientation during assembly. PORMAKE fits the fragments to the `sql` net, and the resulting framework contains the bonded atoms at the junctions.

Manual He-marked XYZ files are the supported input for this release because they let the chemist specify the intended bonding sites and geometry explicitly. Use `0_node/ILCOF-1_node.xyz` and `0_linker/ILCOF-1_linker.xyz` here. For other molecules, supply suitable prepared XYZs in those folders or set `input_nodes` and `input_linkers`. A future input-preparation guide will cover how to place markers. If you override a stage’s output path, give its new location to the stage that reads it. [PORMAKE](https://github.com/Sangwon91/PORMAKE)

Inspect the assembled connectivity in `ILCOF-1/1_ILCOF-1_single_layer/ILCOF-1_unopt.cif`.

In [ ]:
cl.BuildCOF2D().build(TOPOLOGY, COF_NAME)

## Preoptimise the isolated layer

Omitting `topo` lets the full cell relax, including its angles and height. This example explicitly selects `topo=TOPOLOGY` to hold the SQL cell angles at 90° and the out-of-plane height at 15 Å. The in-plane lengths *a* and *b* relax independently. With `fix_z=True`, atoms can move in the plane while their Cartesian *z* coordinates stay fixed. This prepares the layer used to compare stacking arrangements.

The optimiser aims for a maximum force of 0.03 eV/Å on the allowed atom and cell degrees of freedom, with a limit of 2,000 steps. A tighter threshold can require more force evaluations. The fixed-angle procedure took about 48 minutes in one initial laptop CPU run. If the limit is reached, the package raises an error and keeps the intermediate `ILCOF-1_preopt.cif` in the single-layer folder for inspection. Continue with the converged file. [ASE structure optimisation](https://docs.ase-lib.org/ase/optimize.html)

In [ ]:
cl.GeometryOpt(calculator=calculator, max_steps=2000).run_preopt(
    COF_NAME, topo=TOPOLOGY, fmax=0.03, fix_z=True
)

## Sample layer separation and lateral slip

**Interlayer distance (ILD)** is the spacing between adjacent layers. **Interlayer slip (ILS)** is the length of their relative in-plane displacement. AA starts with layers directly above one another. AB denotes a staggered registry, with one layer displaced laterally relative to the next. The atomic alignment depends on the slip direction. In a **serrated** stack successive layer displacements alternate in sign. In an **inclined** stack the displacement repeats in the same direction.

This example explicitly selects `slip_path="a/2"` along the first lattice vector. Omitting `slip_path` uses the SQL diagonal, `"(a+b)/2"`. Choose `"b/2"` to scan along the second lattice vector. The direction and the maximum sampled distance are independent. This scan stops at 6 Å, short of the selected `a/2` endpoint for this example.

Four ILDs (3.2, 3.4, 3.6 and 3.8 Å) and seven ILSs (0 through 6 Å) give **28 serrated CIFs** in `ILCOF-1/2_ILCOF-1_matrix/serr/`. Finer increments, longer paths or a second stacking mode increase the number of structures. This grid illustrates one section of the landscape.

In [ ]:
cl.CreateMatrix(
    ild_start=3.2,
    ild_end=3.8,
    ild_step=0.2,
    ils_length_start=0.0,
    ils_length_end=6.0,
    ils_length_step=1.0,
    slip_path="a/2",
).run(COF_NAME, topo=TOPOLOGY, mode="serr")

## Calculate the energy at each grid point

A single-point calculation evaluates a structure with its positions and cell held fixed. Keeping the imposed ILD and ILS lets us compare the same two variables across the grid. Each added configuration costs another energy evaluation, and more atoms make each evaluation more expensive.

`ILCOF-1/3_ILCOF-1_landscape/ILCOF-1_sp_energies_serr.csv` records the energy of each serrated bilayer cell in eV. Its relative energy is the difference from the lowest successful energy in this scan.

In [ ]:
cl.SinglePoint(calculator=calculator).run(COF_NAME, mode="serr")

## Read the stacking energy landscape

Read lateral slip along the horizontal axis and layer spacing along the vertical axis. The colour shows relative energy in eV, and the marker identifies the lowest sampled configuration. The surface between grid points is a visual guide. A finer grid, a different direction or relaxation could expose another minimum.

`show=True` displays the figure here and saves `pes_ILCOF-1_serr.png` in `ILCOF-1/3_ILCOF-1_landscape/`.

In [ ]:
cl.Landscape().run(COF_NAME, mode="serr", show=True)

## Select the sampled minimum

Select the global minimum **of this grid** as the starting geometry for relaxation. A local minimum is favourable relative to its sampled neighbours. A manual choice lets you examine a particular (ILD, ILS) pair even if it is higher in energy. Optional examples for both choices appear below.

The selected matrix CIF is copied to `ILCOF-1/3_ILCOF-1_landscape/selection/serr/`.

In [ ]:
cl.SelectCofs().run(COF_NAME, mode="serr", autoselect_minima="global")

## Relax the selected stack

Release the single-layer constraints and optimise all atomic positions and cell parameters together. The neighbouring layer can now affect both sheets, so the final ILD and ILS can move away from their starting grid point. The final force threshold is 0.01 eV/Å with a limit of 2,000 steps.

Find the CIF in `ILCOF-1/4_ILCOF-1_optimization/serr/`. Check `stopped_due_to_max_steps` in `ILCOF-1_opt_energies_per_layer.csv` in the parent folder before interpreting it. A true flag identifies an intermediate geometry that still needs convergence. Satisfying the force criterion establishes local numerical convergence.

In [ ]:
cl.GeometryOpt(calculator=calculator, fmax=0.01, max_steps=2000).run(
    COF_NAME, mode="serr"
)

## Inspect the relaxed geometry

The displayed table gives stacking mode, filename, and ILD and ILS in Å to two decimal places. Final serrated energies are reported **per layer**. Relative energies use the lowest energy per layer among the final candidates as their reference. With only one candidate its relative energy is zero by definition.

Jupyter displays the returned table. Full-precision ILD and ILS and the per-layer energies are saved to `ILCOF-1/5_ILCOF-1_analysis/final_structures.csv`.

In [ ]:
cl.AnalyzeStacking().analyze(COF_NAME, mode="serr")

## Simulate powder diffraction

Use Cu Kα₁ (`CuKa1`, about 1.5406 Å) to compare with the supplied copper-radiation scan. The simulated positions and intensities come from an ideal periodic structure. The continuous XY file preserves the individual Bragg reflections and inserts zero-intensity plotting points on a 0.01° grid and at ±0.008° around each reflection. The narrow lines are a plotting convention with no physical peak broadening.

`export_hkl=True` saves a two-row `<structure>_hkl.csv` beside each `.xy` in `ILCOF-1/5_ILCOF-1_analysis/pxrd_xy/serr/`. The first row gives 2θ positions for the ten strongest reflections, sorted by angle. The second gives all contributing Miller-index families for each position. Set `max_reflections` on `run` to export a different count.

In [ ]:
cl.PXRD(wavelength="CuKa1").run(COF_NAME, mode="serr", export_hkl=True)

## Compare the simulated and experimental traces

The plot reads the single experimental file in `experimental_pxrd/` and the simulated XY from the analysis folder. `xlim` controls the visible 2θ window in degrees. To compare beyond the simulated range (1.5–30° by default), extend `two_theta_range` in the simulation cell and regenerate the XY.

The simulated intensity is rescaled over the displayed window. Compare peak positions and relative intensities while allowing for preferred orientation, disorder and instrumental effects. Different layer arrangements can yield similar powder patterns, so agreement alone cannot uniquely establish stacking. The figure is displayed here and saved as a PDF in `ILCOF-1/5_ILCOF-1_analysis/pxrd_plots/serr/`.

In [ ]:
cl.PXRD().plot_sim_vs_exp(COF_NAME, mode="serr", xlim=(2, 30), show=True)

## Optional analyses

The following examples are Markdown. Copy a block into a code cell only when you want that analysis.

### Screen the full stacking matrix by PXRD

Once the matrix exists, compare the experimental scan with each of the 28 unrelaxed configurations. Inspect how features change along slip and spacing, including configurations away from the sampled energy minimum. The four-page `ILCOF-1/3_ILCOF-1_landscape/pxrd/serr/ILCOF-1_serr.pdf` groups up to nine comparisons per page, ordered by ILS then ILD. Screening leaves the choice of matching configurations to you.

```python
cl.PXRD(wavelength="CuKa1").screen_landscape(COF_NAME, mode="serr")
```

### Measure peak-position differences

After inspecting the comparison, edit the two example windows below to bracket corresponding experimental and simulated features. Each interval is in degrees 2θ. Extraction fits the experimental peak centre and retains simulated reflections at or above 10% of the strongest intensity in that region. It writes `<structure>_regions.csv` to `5_ILCOF-1_analysis/pxrd_peaks/serr/` under the COF folder.

Call the diagnostic on the same object. **Δ2θ = simulated centroid − experimental centre**. Positive values put the calculated reflection at a higher angle. The simulated centroid is intensity-weighted within each region. Matched regions have equal weight in the summary count, mean, minimum and maximum. An unmatched region stays in the first table with a missing offset. Inspect both tables without altering the structure.

```python
diagnostics = cl.PXRD(wavelength="CuKa1")
diagnostics.extract_peak_regions(
    COF_NAME, mode="serr", peak_regions=[(3.3, 4.3), (7.1, 8.1)]
)
regions, summary = diagnostics.peak_position_diagnostics()
print(regions.to_string(index=False))
summary
```

### Estimate geometric porosity

Install the optional extra in the notebook environment with `pip install "cof-landscaper[porosity]"`. The default uses a spherical probe of radius 1.86 Å, representative of nitrogen, to determine which channels and surfaces it can reach.

The returned table includes accessible surface area (m²/g and m²/cm³), accessible volume for the probe centre (cm³/g), and accessible void fraction (percentage of the cell volume). Pore-limiting diameter (Å) is the diameter of the largest sphere that can traverse a continuous channel. These quantities describe the supplied periodic geometry. Experimental BET area is inferred from gas adsorption and also depends on the adsorbate, sample activation and accessibility. A CSV is saved in `ILCOF-1/5_ILCOF-1_analysis/porosity/serr/`.

```python
cl.Porosity().run(COF_NAME, mode="serr")
```

### Select other starting structures

Use either example after the energy calculation. Separate selection folders keep these CIFs apart from the main workflow’s global minimum. To optimise one of these sets, pass its selection root as `input_base` to `GeometryOpt.run` and use a separate `output_base`.

Select detected local minima on the sampled grid:

```python
cl.SelectCofs().run(
    COF_NAME, mode="serr", autoselect_minima="local",
    output_base=f"{COF_NAME}/3_{COF_NAME}_landscape/selection_local",
)
```

Select a specific sampled pair, here ILD = 3.4 Å and ILS = 2.0 Å. Disabling autoselection selects only the requested pair.

```python
cl.SelectCofs().run(
    COF_NAME, mode="serr", selections_serr=[(3.4, 2.0)], include_autoselect=False,
    output_base=f"{COF_NAME}/3_{COF_NAME}_landscape/selection_manual",
)
```